In [ ]:
import polars as pl
import glob

In [ ]:
# Configuration
INPUT_PATTERN = "PATH_TO_FILE"  # Adjust path/pattern
OUTPUT_FILE = "PATH_TO_FILE"

# 1. Get list of files manually
files = glob.glob(INPUT_PATTERN)
print(f"Found {len(files)} files.")

# 2. Create a list of LazyFrames
#    This builds a plan to read all files, but doesn't load them yet.
lfs = [pl.scan_parquet(f) for f in files]

# 3. Concatenate with relaxation
#    how="vertical_relaxed" allows slight type mismatches (like int32 vs int64)
#    how="diagonal" allows missing columns (fills with nulls)
combined_lazy = pl.concat(lfs, how="vertical_relaxed")

# 4. Stream to disk
print("Streaming to disk...")
combined_lazy.sink_parquet(OUTPUT_FILE, engine='streaming')
print("Done.")

## Check outfile

In [ ]:
of = pl.scan_parquet(OUTPUT_FILE)
of.head().collect()

In [ ]:
(
    of
    .with_columns(
        POS = pl.col('POS').cast(pl.Int64)
    )
    .sink_parquet("PATH_TO_FILE", engine='streaming')
)

In [ ]:
!dx upload PATH_TO_FILE --path project-REDACTED:/processed_data/wgs/